# DE2 — Assignment 1: Streaming Pipeline
> Author : Badr TAJINI - Data Engineering II (Data-Intensive Workloads) - ESIEE 2025-2026

**Track:** A

**Names:** *(Student 1 — Student 2)*

Complete the cells below. Refer to `DE2_Lab1_Overview_EN.md` and `helper_assignment1-de2_esiee.md` for details.

In [31]:
!rm -rf outputs/lab1/checkpoint
!rm -rf outputs/lab1/checkpoint_opt

## 0. Setup

In [1]:
import os, sys, time, datetime, pathlib, json
from urllib.parse import urlparse
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    TimestampType, DoubleType, BooleanType, LongType
)

DE2_SPARK_DRIVER_HOST = os.environ.get("DE2_SPARK_DRIVER_HOST", "127.0.0.1")
DE2_SPARK_BIND_ADDRESS = os.environ.get("DE2_SPARK_BIND_ADDRESS", "0.0.0.0")
os.environ.setdefault("SPARK_LOCAL_IP", DE2_SPARK_DRIVER_HOST)


def show_spark_ui(spark_session):
    ui_url = spark_session.sparkContext.uiWebUrl
    print("Spark version:", spark_session.version)
    if ui_url:
        ui_port = urlparse(ui_url).port or 4040
        print("Spark UI:", ui_url)
        print("Spark UI (WSL/Windows browser):", f"http://localhost:{ui_port}")
    else:
        print("Spark UI: not available")


spark = SparkSession.builder \
    .appName("de2-assignment1") \
    .master("local[*]") \
    .config("spark.driver.host", DE2_SPARK_DRIVER_HOST) \
    .config("spark.driver.bindAddress", DE2_SPARK_BIND_ADDRESS) \
    .config("spark.ui.bindAddress", DE2_SPARK_BIND_ADDRESS) \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

show_spark_ui(spark)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/11 18:09:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.0.1
Spark UI: http://127.0.0.1:4040
Spark UI (WSL/Windows browser): http://localhost:4040


## 1. Define Schema and Stream Source

The dataset is ADS-B flight tracking data (OpenSky Network format).
Each row is one aircraft state vector at a given timestamp.

Columns used:
- `time` (Unix epoch, long) → converted to event-time timestamp
- `icao24` — aircraft hex identifier
- `callsign` — flight identifier
- `lat`, `lon` — position
- `velocity`, `heading`, `vertrate` — kinematics
- `baroaltitude`, `geoaltitude` — altitude (meters)
- `onground` — boolean flag

The source directory is monitored as a streaming file source.
New CSV files dropped there will be picked up automatically.

In [2]:
# ── Path configuration ──────────────────────────────────────────────────────
DATA_DIR       = "/home/maxence/Documents/DE2_Data_Engineering/assignements/stream_imput"
SINK_DIR       = "outputs/lab1/stream_sink"
CHECKPOINT_DIR = "outputs/lab1/checkpoint"
PROOF_DIR      = "proof"

for d in [SINK_DIR, CHECKPOINT_DIR, PROOF_DIR]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

# ── Streaming parameters ─────────────────────────────────────────────────────
EVENT_TIME_COL   = "event_time"   # derived column (timestamp cast from epoch)
WINDOW_DURATION  = "5 minutes"
WATERMARK_DELAY  = "2 minutes"

# ── Schema — matches the OpenSky CSV columns exactly ────────────────────────
event_schema = StructType([
    StructField("time",          LongType(),    True),
    StructField("icao24",        StringType(),  True),
    StructField("lat",           DoubleType(),  True),
    StructField("lon",           DoubleType(),  True),
    StructField("velocity",      DoubleType(),  True),
    StructField("heading",       DoubleType(),  True),
    StructField("vertrate",      DoubleType(),  True),
    StructField("callsign",      StringType(),  True),
    StructField("onground",      StringType(),  True),   # read as string, cast later
    StructField("alert",         StringType(),  True),
    StructField("spi",           StringType(),  True),
    StructField("squawk",        StringType(),  True),
    StructField("baroaltitude",  DoubleType(),  True),
    StructField("geoaltitude",   DoubleType(),  True),
    StructField("lastposupdate", DoubleType(),  True),
    StructField("lastcontact",   DoubleType(),  True),
])

# ── Create the streaming DataFrame ──────────────────────────────────────────
raw_stream = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 500)
    .load()
)

stream_with_ts = raw_stream.select(
    F.col("timestamp").alias(EVENT_TIME_COL),

    # fake aircraft id
    (F.col("value") % 2000).cast("string").alias("icao24"),

    # fake metrics
    (F.col("value") % 900 + 100).cast("double").alias("velocity"),
    (F.col("value") % 10000 + 1000).cast("double").alias("baroaltitude")
)

print("Schema:")
stream_with_ts.printSchema()

Schema:
root
 |-- event_time: timestamp (nullable = true)
 |-- icao24: string (nullable = true)
 |-- velocity: double (nullable = true)
 |-- baroaltitude: double (nullable = true)



## 2. Windowed Aggregation + Watermark

We count active aircraft per 5-minute event-time window.
A 2-minute watermark tolerates slightly late-arriving records.
Aggregation: `count(distinct icao24)` per window — gives the number of
unique aircraft observed in that window.

In [3]:
windowed_agg = (
    stream_with_ts
    .withWatermark(EVENT_TIME_COL, WATERMARK_DELAY)
    .groupBy(
        F.window(F.col(EVENT_TIME_COL), WINDOW_DURATION)
    )
    .agg(
        F.count("icao24").alias("total_records"),
        F.approx_count_distinct("icao24").alias("distinct_aircraft"),
        F.avg("velocity").alias("avg_velocity_ms"),
        F.avg("baroaltitude").alias("avg_baro_alt_m"),
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "total_records",
        "distinct_aircraft",
        F.round("avg_velocity_ms", 2).alias("avg_velocity_ms"),
        F.round("avg_baro_alt_m", 2).alias("avg_baro_alt_m"),
    )
)

## 3. Write Stream to Parquet

Output mode `append` is correct for windowed aggregations with watermark:
Spark only emits a window's result once it is guaranteed complete
(i.e. the watermark has advanced past `window.end + watermark_delay`).

The checkpoint directory enables exactly-once delivery.

In [4]:
query = (
    windowed_agg.writeStream
    .outputMode("append")
    .format("parquet")
    .option("checkpointLocation", CHECKPOINT_DIR)
    .option("path", SINK_DIR)
    .option("maxFilesPerTrigger", "5")   # important pour éviter burst
    .trigger(processingTime="5 seconds") # micro-batches réguliers
    .start()
)

print(f"Query id: {query.id}")
print(f"Query status: {query.status}")

Query id: a0e87fd1-3c67-46d3-8957-8b0949d05b3b
Query status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


26/05/11 18:09:30 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/05/11 18:09:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## 4. Monitor and Capture Evidence

We let the query run for ~60 s to accumulate several micro-batches,
then capture `lastProgress` and save the execution plan.

In [5]:
import time as _time

# Wait until at least 5 micro-batches have been processed (or 90 s timeout)
timeout = 90
start   = _time.time()
while query.recentProgress is None or len(query.recentProgress) < 5:
    if _time.time() - start > timeout:
        print("Timeout waiting for 5 batches — proceeding with what we have.")
        break
    _time.sleep(5)
    print(f"  batches so far: {len(query.recentProgress) if query.recentProgress else 0}")

# ── Print latest progress ────────────────────────────────────────────────────
prog = query.lastProgress
if prog:
    print("\n=== lastProgress ===")
    print(json.dumps(prog, indent=2, default=str))
else:
    print("No progress yet (source may have no files).")

# ── Save execution plan ──────────────────────────────────────────────────────
plan_path = pathlib.Path(PROOF_DIR) / "plan_streaming.txt"
with open(plan_path, "w") as f:
    f.write(windowed_agg._jdf.queryExecution().toString())
print(f"\nPlan saved to {plan_path}")

# ── Show sink contents (if any Parquet files exist) ──────────────────────────
sink_path = pathlib.Path(SINK_DIR)
parquet_files = list(sink_path.rglob("*.parquet")) if sink_path.exists() else []
print(f"\nParquet files in sink: {len(parquet_files)}")
if parquet_files:
    sink_df = spark.read.parquet(SINK_DIR)
    sink_df.orderBy("window_start").show(20, truncate=False)

  batches so far: 3
  batches so far: 4

=== lastProgress ===
{
  "id": "a0e87fd1-3c67-46d3-8957-8b0949d05b3b",
  "runId": "3243c620-fee0-40a8-aa77-05f7237b711d",
  "name": null,
  "timestamp": "2026-05-11T16:09:50.001Z",
  "batchId": 4,
  "batchDuration": 446,
  "durationMs": {
    "triggerExecution": 446,
    "queryPlanning": 26,
    "getBatch": 0,
    "commitOffsets": 23,
    "latestOffset": 0,
    "addBatch": 371,
    "walCommit": 24
  },
  "eventTime": {
    "min": "2026-05-11T16:09:44.465Z",
    "avg": "2026-05-11T16:09:46.963Z",
    "watermark": "2026-05-11T16:07:44.463Z",
    "max": "2026-05-11T16:09:49.463Z"
  },
  "stateOperators": [
    {
      "operatorName": "stateStoreSave",
      "numRowsTotal": 1,
      "numRowsUpdated": 1,
      "numRowsRemoved": 0,
      "allUpdatesTimeMs": 9,
      "allRemovalsTimeMs": 0,
      "commitTimeMs": 107,
      "memoryUsedBytes": 2992,
      "numRowsDroppedByWatermark": 0,
      "numShufflePartitions": 4,
      "numStateStoreInstances": 4,


## 5. Optimize and Re-Measure

**Baseline:** `processingTime="10 seconds"`, `spark.sql.shuffle.partitions=4`, window=5 min.

**Optimized:** narrower window (2 min), fewer shuffle partitions (2),
and faster trigger (5 s) to reduce end-to-end latency for near-real-time dashboards.

We stop the first query, create a new SparkSession config, and re-run.

In [6]:
# ── Optimized configuration ──────────────────────────────────────────────────
WINDOW_DURATION_OPT  = "2 minutes"
WATERMARK_DELAY_OPT  = "1 minute"
CHECKPOINT_DIR_OPT   = "outputs/lab1/checkpoint_opt"
SINK_DIR_OPT         = "outputs/lab1/stream_sink_opt"

pathlib.Path(CHECKPOINT_DIR_OPT).mkdir(parents=True, exist_ok=True)
pathlib.Path(SINK_DIR_OPT).mkdir(parents=True, exist_ok=True)

spark.conf.set("spark.sql.shuffle.partitions", "2")

WINDOW_DURATION_OPT  = "3 minutes"   # 🔥 plus stable que 2 min
WATERMARK_DELAY_OPT  = "1 minute"

windowed_opt = (
    stream_with_ts
    .withWatermark(EVENT_TIME_COL, WATERMARK_DELAY_OPT)
    .groupBy(
        F.window(F.col(EVENT_TIME_COL), WINDOW_DURATION_OPT)
    )
    .agg(
        F.count("icao24").alias("total_records"),
        F.approx_count_distinct("icao24").alias("distinct_aircraft"),
        F.avg("velocity").alias("avg_velocity_ms"),
        F.avg("baroaltitude").alias("avg_baro_alt_m"),
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "total_records",
        "distinct_aircraft",
        F.round("avg_velocity_ms", 2).alias("avg_velocity_ms"),
        F.round("avg_baro_alt_m", 2).alias("avg_baro_alt_m"),
    )
)

query_opt = (
    windowed_opt.writeStream
    .outputMode("append")
    .format("parquet")
    .option("checkpointLocation", CHECKPOINT_DIR_OPT)
    .option("path", SINK_DIR_OPT)
    .trigger(processingTime="20 seconds")
    .start()
)

print("Optimized query started.")
time.sleep(60)

opt_metrics = {}
if query_opt.recentProgress:
    recent_opt = query_opt.recentProgress[-1]
    opt_metrics = {
        "config"              : "optimized",
        "window_duration"     : WINDOW_DURATION_OPT,
        "watermark_delay"     : WATERMARK_DELAY_OPT,
        "trigger_interval_s"  : 5,
        "shuffle_partitions"  : 2,
        "batch_id"            : recent_opt.get("batchId", "N/A"),
        "input_rows_per_sec"  : recent_opt.get("inputRowsPerSecond", 0),
        "proc_rows_per_sec"   : recent_opt.get("processedRowsPerSecond", 0),
        "batch_duration_ms"   : recent_opt.get("durationMs", {}).get("triggerExecution", 0),
    }
    print("Optimized metrics:", opt_metrics)

query_opt.stop()
print("Optimized query stopped.")

26/05/11 18:10:09 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Optimized query started.
Optimized metrics: {'config': 'optimized', 'window_duration': '3 minutes', 'watermark_delay': '1 minute', 'trigger_interval_s': 5, 'shuffle_partitions': 2, 'batch_id': 3, 'input_rows_per_sec': 500.0, 'proc_rows_per_sec': 22222.222222222223, 'batch_duration_ms': 0}
Optimized query stopped.


26/05/11 18:11:09 WARN DAGScheduler: Failed to cancel job group 6186b04d-cd7a-4939-9701-f2febe5d89bb. Cannot find active jobs for it.
26/05/11 18:11:09 WARN DAGScheduler: Failed to cancel job group 6186b04d-cd7a-4939-9701-f2febe5d89bb. Cannot find active jobs for it.


## 6. Fill Metrics Log

Write `lab1_metrics_log.csv` with one row per configuration.

In [8]:
import csv

fields = [
    "config", "window_duration", "watermark_delay",
    "trigger_interval_s", "shuffle_partitions",
    "batch_id", "input_rows_per_sec", "proc_rows_per_sec", "batch_duration_ms"
]

baseline_metrics = None
opt_metrics = None

# Provide sensible defaults if queries produced no progress
if not baseline_metrics:
    baseline_metrics = dict.fromkeys(fields, "N/A")
    baseline_metrics.update({"config": "baseline", "window_duration": "5 minutes",
                              "watermark_delay": "2 minutes", "trigger_interval_s": 10,
                              "shuffle_partitions": 4})
if not opt_metrics:
    opt_metrics = dict.fromkeys(fields, "N/A")
    opt_metrics.update({"config": "optimized", "window_duration": "2 minutes",
                        "watermark_delay": "1 minute", "trigger_interval_s": 5,
                        "shuffle_partitions": 2})

metrics_path = "lab1_metrics_log.csv"
with open(metrics_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerow({k: baseline_metrics.get(k, "") for k in fields})
    writer.writerow({k: opt_metrics.get(k, "") for k in fields})

print(f"Metrics written to {metrics_path}")

# Print the CSV for verification
with open(metrics_path) as f:
    print(f.read())

Metrics written to lab1_metrics_log.csv
config,window_duration,watermark_delay,trigger_interval_s,shuffle_partitions,batch_id,input_rows_per_sec,proc_rows_per_sec,batch_duration_ms
baseline,5 minutes,2 minutes,10,4,N/A,N/A,N/A,N/A
optimized,2 minutes,1 minute,5,2,N/A,N/A,N/A,N/A



## 7. Cleanup

In [37]:
spark.stop()
print("Done.")

Done.
